# Course Architect — an agent that turns an outline into a finished PowerPoint course

**5-Day AI Agents Intensive (Vibe Coding) — Capstone · Track: Agents for Business**

Building courseware is slow: someone turns a rough outline into dozens of well-structured,
on-brand slides with speaker notes. **Course Architect** is an AI agent that does it — you
give it a short outline (a title, an audience, a few modules with bullet points) — or even
just a topic, and it designs the outline itself — then it plans the course, writes every
slide and speaker note, assembles a branded `.pptx`, and **checks its own work**: did every
point actually make it onto a slide?

This is a real-world business agent, distilled from a production course-generation system.
It runs on **Gemini**, and degrades gracefully to a deterministic offline mode if no key
is present — so this notebook always produces a real deck.

### What it demonstrates (one per course day)
| Day | Concept | Where |
|----|----|----|
| 1 | Vibe coding — NL roles drive the work | Planner & Writer prompts in `agent/provider.py` |
| 2 | Tools & interoperability (MCP) | `agent/tools.py` — `assemble_deck`, `check_coverage` |
| 3 | Context engineering: skills & memory | `agent/skills/*.md`, multi-turn `agent/memory.py` |
| 4 | Agent quality & evaluation | coverage self-check drives self-correction (`builder.py`) |
| 5 | Prototype → production | model writes content / code only assembles; reproducible; graceful |

> **Runs with or without a key.** Add a Kaggle Secret `GOOGLE_API_KEY` (AI Studio) to have
> **Gemini** write the content. Without one, a deterministic offline writer fills the deck so
> the notebook still runs end-to-end and emits a real `.pptx`.


## 1. Install deps & write out the project
`python-pptx` builds the slides; `google-genai` powers the (optional) Gemini path.
Each module is then written from its own cell, so the code below is exactly what runs.

In [ ]:
import importlib, subprocess, sys
def ensure(pkg, mod):
    try:
        importlib.import_module(mod); return True
    except ImportError:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg]); return True
        except Exception as e:
            print("could not install", pkg, "-", e); return False
ensure("python-pptx", "pptx")
ensure("google-genai", "google.genai")  # only needed for the Gemini path

import os
for d in ("deck", "agent", "agent/skills"):
    os.makedirs(d, exist_ok=True)
print("ready")

In [ ]:
%%writefile deck/__init__.py
"""Course Architect — deck domain layer.

The *content contract* between the agent and the file it produces. The agent (LLM)
generates a `DeckSpec`; the assembler turns it into a real `.pptx`. This separation
is the hard-won EQV production lesson: **the model produces all the content; code
only assembles it** — code never invents slide text.
"""

from .spec import (
    Slide, Section, DeckSpec, Outline,
    parse_outline, coverage_report,
)
from .theme import THEMES, get_theme, Theme
from .assembler import assemble_pptx

__all__ = [
    "Slide", "Section", "DeckSpec", "Outline",
    "parse_outline", "coverage_report",
    "THEMES", "get_theme", "Theme",
    "assemble_pptx",
]


In [ ]:
%%writefile deck/spec.py
"""The content contract: outline in, deck spec out, plus a coverage evaluator.

* `Outline`   — the short input the user provides (title, audience, modules+bullets).
* `DeckSpec`  — the full deck the agent produces (sections of slides). Code assembles
                this verbatim; it never adds content of its own.
* `coverage_report` — the agent's objective self-check: did every input bullet make it
                onto a slide? This enforces the EQV rule "cover ONLY the bullets given"
                and is the Day-4 evaluation signal that drives self-correction.
"""

from __future__ import annotations

import re
from dataclasses import asdict, dataclass, field
from typing import Dict, List


# --------------------------------------------------------------------------- #
# Input outline
# --------------------------------------------------------------------------- #

@dataclass
class Outline:
    title: str
    audience: str
    modules: List[Dict]  # [{"title": str, "bullets": [str]}]

    def all_bullets(self) -> List[str]:
        out: List[str] = []
        for m in self.modules:
            out.extend(m.get("bullets", []))
        return out


def parse_outline(text: str) -> Outline:
    """Parse a tiny markdown-ish outline. Format:

        # Course Title
        Audience: Frontline managers (beginner)
        ## Module One Title
        - first point
        - second point
        ## Module Two Title
        - ...
    """
    title, audience = "Untitled Course", "General audience"
    modules: List[Dict] = []
    current: Dict | None = None
    for raw in text.splitlines():
        line = raw.rstrip()
        if not line.strip():
            continue
        if line.startswith("## "):
            current = {"title": line[3:].strip(), "bullets": []}
            modules.append(current)
        elif line.startswith("# "):
            title = line[2:].strip()
        elif re.match(r"(?i)^audience\s*:", line):
            audience = line.split(":", 1)[1].strip()
        elif line.lstrip().startswith(("-", "*", "•")):
            bullet = line.lstrip()[1:].strip()
            if bullet and current is not None:
                current["bullets"].append(bullet)
        # bare lines before any module are ignored (e.g. a description)
    if not modules:
        # Fall back: treat every bullet/line as a single module.
        modules = [{"title": title, "bullets":
                    [l.strip("-*• ").strip() for l in text.splitlines()
                     if l.strip() and not l.startswith("#")]}]
    return Outline(title=title, audience=audience, modules=modules)


# --------------------------------------------------------------------------- #
# Output deck
# --------------------------------------------------------------------------- #

@dataclass
class Slide:
    title: str
    bullets: List[str] = field(default_factory=list)
    notes: str = ""                       # speaker notes
    example: str = ""                     # one concrete worked example (rendered in a box)
    covers: List[str] = field(default_factory=list)  # which input bullets this slide addresses


@dataclass
class Section:
    module_title: str
    objective: str = ""
    slides: List[Slide] = field(default_factory=list)


@dataclass
class DeckSpec:
    title: str
    subtitle: str = ""
    audience: str = ""
    theme: str = "corporate"
    sections: List[Section] = field(default_factory=list)

    # -- (de)serialisation for memory + LLM JSON --------------------------- #
    def to_dict(self) -> Dict:
        return {
            "title": self.title, "subtitle": self.subtitle,
            "audience": self.audience, "theme": self.theme,
            "sections": [
                {"module_title": s.module_title, "objective": s.objective,
                 "slides": [asdict(sl) for sl in s.slides]}
                for s in self.sections],
        }

    @staticmethod
    def from_dict(d: Dict) -> "DeckSpec":
        sections = []
        for s in d.get("sections", []):
            slides = [Slide(title=sl.get("title", ""),
                            bullets=list(sl.get("bullets", [])),
                            notes=sl.get("notes", ""),
                            example=sl.get("example", ""),
                            covers=list(sl.get("covers", [])))
                      for sl in s.get("slides", [])]
            sections.append(Section(module_title=s.get("module_title", ""),
                                    objective=s.get("objective", ""), slides=slides))
        return DeckSpec(title=d.get("title", "Untitled"), subtitle=d.get("subtitle", ""),
                        audience=d.get("audience", ""), theme=d.get("theme", "corporate"),
                        sections=sections)

    def content_slide_count(self) -> int:
        return sum(len(s.slides) for s in self.sections)


# --------------------------------------------------------------------------- #
# Coverage evaluation (the self-check)
# --------------------------------------------------------------------------- #

def _norm(s: str) -> str:
    return re.sub(r"[^a-z0-9 ]", "", s.lower()).strip()


def _tokens(s: str) -> set:
    return set(_norm(s).split())


def coverage_report(outline: Outline, deck: DeckSpec) -> Dict:
    """Was every input bullet represented on a slide?

    A bullet counts as covered if it's listed in a slide's `covers`, or if a slide's
    title/bullets overlap it strongly (token Jaccard >= 0.5) — so we don't punish the
    model for paraphrasing while still catching genuinely dropped material.
    """
    # Gather all slide text + explicit covers.
    explicit = set()
    slide_blobs: List[set] = []
    for sec in deck.sections:
        for sl in sec.slides:
            for c in sl.covers:
                explicit.add(_norm(c))
            slide_blobs.append(_tokens(sl.title + " " + " ".join(sl.bullets)))

    per_bullet: Dict[str, bool] = {}
    missing: List[str] = []
    for bullet in outline.all_bullets():
        nb = _norm(bullet)
        covered = nb in explicit
        if not covered:
            bt = _tokens(bullet)
            if bt:
                for blob in slide_blobs:
                    inter = len(bt & blob)
                    union = len(bt | blob) or 1
                    if inter / union >= 0.5 or (len(bt) >= 2 and bt <= blob):
                        covered = True
                        break
        per_bullet[bullet] = covered
        if not covered:
            missing.append(bullet)

    total = len(per_bullet)
    covered_n = total - len(missing)
    return {
        "total_bullets": total,
        "covered": covered_n,
        "missing": missing,
        "score": round(covered_n / total, 3) if total else 1.0,
        "per_bullet": per_bullet,
    }


In [ ]:
%%writefile deck/theme.py
"""Visual themes (branding applied *last*, to a finished content table).

Generic, clean corporate looks — deliberately NOT any real client's brand assets.
Swapping the active theme restyles the whole deck without touching content, which
is the "template + content table, brand last not first" approach from EQV.
"""

from __future__ import annotations

from dataclasses import dataclass


@dataclass(frozen=True)
class Theme:
    name: str
    bg: str             # slide background (hex, no #)
    band: str           # accent band / section background
    title: str          # title text colour
    body: str           # body text colour
    muted: str          # subtitle / footer colour
    accent: str         # bullet marker / rule colour
    font_title: str = "Calibri"
    font_body: str = "Calibri"


THEMES = {
    "corporate": Theme(
        name="corporate", bg="FFFFFF", band="1F3A5F", title="1F3A5F",
        body="222831", muted="6B7280", accent="2E86C1",
        font_title="Calibri", font_body="Calibri"),
    "slate": Theme(
        name="slate", bg="0F172A", band="1E293B", title="F8FAFC",
        body="E2E8F0", muted="94A3B8", accent="38BDF8",
        font_title="Calibri", font_body="Calibri"),
    "fresh": Theme(
        name="fresh", bg="FFFFFF", band="0E7C66", title="0E7C66",
        body="1F2937", muted="6B7280", accent="F59E0B",
        font_title="Calibri", font_body="Calibri"),
}


def get_theme(name: str) -> Theme:
    return THEMES.get((name or "").lower(), THEMES["corporate"])


In [ ]:
%%writefile deck/assembler.py
"""The assembler tool: a DeckSpec becomes a real .pptx via python-pptx.

This is pure layout. It reads the content the agent produced and lays it out on
16:9 slides with the chosen theme. It adds NO content of its own — if a bullet
isn't in the spec, it won't appear in the deck. (That property is what the coverage
evaluator checks.)
"""

from __future__ import annotations

from typing import List

from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from pptx.enum.text import MSO_ANCHOR, PP_ALIGN
from pptx.util import Inches, Pt

from .spec import DeckSpec, Section, Slide
from .theme import Theme, get_theme

_EMU_W, _EMU_H = Inches(13.333), Inches(7.5)  # 16:9


def _rgb(hex6: str) -> RGBColor:
    return RGBColor(int(hex6[0:2], 16), int(hex6[2:4], 16), int(hex6[4:6], 16))


def _blank(prs: Presentation):
    return prs.slides.add_slide(prs.slide_layouts[6])  # 6 = blank


def _set_bg(slide, hex6: str) -> None:
    fill = slide.background.fill
    fill.solid()
    fill.fore_color.rgb = _rgb(hex6)


def _accent_band(slide, theme: Theme, height_in: float = 0.18, top_in: float = 0.0) -> None:
    box = slide.shapes.add_textbox(0, Inches(top_in), _EMU_W, Inches(height_in))
    box.fill.solid(); box.fill.fore_color.rgb = _rgb(theme.accent)
    box.line.fill.background()


def _textbox(slide, left, top, width, height, anchor=MSO_ANCHOR.TOP):
    box = slide.shapes.add_textbox(left, top, width, height)
    tf = box.text_frame
    tf.word_wrap = True
    tf.vertical_anchor = anchor
    return tf


def _style_run(run, size, color_hex, font, bold=False):
    run.font.size = Pt(size)
    run.font.bold = bold
    run.font.name = font
    run.font.color.rgb = _rgb(color_hex)


def _footer(slide, theme: Theme, text: str) -> None:
    tf = _textbox(slide, Inches(0.4), Inches(7.05), Inches(12.5), Inches(0.35))
    p = tf.paragraphs[0]
    run = p.add_run(); run.text = text
    _style_run(run, 10, theme.muted, theme.font_body)


# --------------------------------------------------------------------------- #

def _title_slide(prs, deck: DeckSpec, theme: Theme) -> None:
    slide = _blank(prs)
    _set_bg(slide, theme.bg)
    _accent_band(slide, theme, height_in=2.6, top_in=2.45)  # central band
    tf = _textbox(slide, Inches(0.9), Inches(2.7), Inches(11.5), Inches(1.5))
    p = tf.paragraphs[0]; p.alignment = PP_ALIGN.LEFT
    r = p.add_run(); r.text = deck.title
    _style_run(r, 40, "FFFFFF", theme.font_title, bold=True)
    if deck.subtitle or deck.audience:
        sub = _textbox(slide, Inches(0.9), Inches(4.15), Inches(11.5), Inches(0.8))
        p2 = sub.paragraphs[0]
        r2 = p2.add_run()
        r2.text = deck.subtitle or f"For: {deck.audience}"
        _style_run(r2, 18, "FFFFFF", theme.font_body)


def _section_slide(prs, section: Section, theme: Theme, idx: int) -> None:
    slide = _blank(prs)
    _set_bg(slide, theme.band)
    tf = _textbox(slide, Inches(0.9), Inches(2.9), Inches(11.5), Inches(1.2),
                  anchor=MSO_ANCHOR.MIDDLE)
    p = tf.paragraphs[0]
    r = p.add_run(); r.text = f"Module {idx}"
    _style_run(r, 16, theme.accent, theme.font_body, bold=True)
    p2 = tf.add_paragraph()
    r2 = p2.add_run(); r2.text = section.module_title
    _style_run(r2, 34, "FFFFFF", theme.font_title, bold=True)
    if section.objective:
        p3 = tf.add_paragraph()
        r3 = p3.add_run(); r3.text = section.objective
        _style_run(r3, 16, "E5E7EB", theme.font_body)


def _content_slide(prs, slide_spec: Slide, theme: Theme, footer: str) -> None:
    slide = _blank(prs)
    _set_bg(slide, theme.bg)
    _accent_band(slide, theme)
    # Title
    ttf = _textbox(slide, Inches(0.6), Inches(0.45), Inches(12.1), Inches(1.0))
    tp = ttf.paragraphs[0]
    tr = tp.add_run(); tr.text = slide_spec.title
    _style_run(tr, 28, theme.title, theme.font_title, bold=True)
    # Bullets (leave room for the example box when present)
    body_h = 3.5 if slide_spec.example else 5.0
    btf = _textbox(slide, Inches(0.7), Inches(1.7), Inches(11.9), Inches(body_h))
    first = True
    for bullet in slide_spec.bullets:
        p = btf.paragraphs[0] if first else btf.add_paragraph()
        first = False
        p.space_after = Pt(10)
        marker = p.add_run(); marker.text = "•  "
        _style_run(marker, 18, theme.accent, theme.font_body, bold=True)
        run = p.add_run(); run.text = bullet
        _style_run(run, 18, theme.body, theme.font_body)
    if not slide_spec.bullets:
        btf.paragraphs[0].add_run().text = ""
    # Worked-example box (content the model produced; layout only here)
    if slide_spec.example:
        box = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE,
                                     Inches(0.7), Inches(5.35), Inches(11.9), Inches(1.5))
        box.fill.solid(); box.fill.fore_color.rgb = _rgb(theme.band)
        box.line.fill.background()
        etf = box.text_frame
        etf.word_wrap = True
        etf.margin_left = Inches(0.25); etf.margin_right = Inches(0.25)
        etf.margin_top = Inches(0.12); etf.margin_bottom = Inches(0.12)
        ep = etf.paragraphs[0]
        lab = ep.add_run(); lab.text = "Example   "
        _style_run(lab, 15, theme.accent, theme.font_body, bold=True)
        er = ep.add_run(); er.text = slide_spec.example
        _style_run(er, 15, "FFFFFF", theme.font_body)
    _footer(slide, theme, footer)
    # Speaker notes
    if slide_spec.notes:
        slide.notes_slide.notes_text_frame.text = slide_spec.notes


def _summary_slide(prs, deck: DeckSpec, theme: Theme) -> None:
    slide = _blank(prs)
    _set_bg(slide, theme.bg)
    _accent_band(slide, theme)
    ttf = _textbox(slide, Inches(0.6), Inches(0.45), Inches(12.1), Inches(1.0))
    tr = ttf.paragraphs[0].add_run(); tr.text = "Course summary"
    _style_run(tr, 28, theme.title, theme.font_title, bold=True)
    btf = _textbox(slide, Inches(0.7), Inches(1.7), Inches(11.9), Inches(5.0))
    first = True
    for i, sec in enumerate(deck.sections, 1):
        p = btf.paragraphs[0] if first else btf.add_paragraph()
        first = False
        p.space_after = Pt(8)
        marker = p.add_run(); marker.text = f"{i}.  "
        _style_run(marker, 18, theme.accent, theme.font_body, bold=True)
        run = p.add_run(); run.text = sec.module_title + (f" — {sec.objective}" if sec.objective else "")
        _style_run(run, 18, theme.body, theme.font_body)


def assemble_pptx(deck: DeckSpec, out_path: str, theme: Theme | None = None) -> str:
    """Build the .pptx and return its path."""
    th = theme or get_theme(deck.theme)
    prs = Presentation()
    prs.slide_width, prs.slide_height = _EMU_W, _EMU_H

    _title_slide(prs, deck, th)
    n = deck.content_slide_count()
    page = 0
    for i, section in enumerate(deck.sections, 1):
        _section_slide(prs, section, th, i)
        for sl in section.slides:
            page += 1
            _content_slide(prs, sl, th, f"{deck.title}   ·   {page}/{n}")
    _summary_slide(prs, deck, th)

    prs.save(out_path)
    return out_path


In [ ]:
%%writefile agent/__init__.py
"""Course Architect — the capstone agent.

A vibe-coded, tool-using agent that turns a short outline into a finished, branded
PowerPoint course. Maps to the 5 course days:
  * provider  — Gemini planner + slide-writer, with a no-key Mock fallback   (D1 vibe coding)
  * tools     — MCP-style assemble_deck / check_coverage                     (D2 tools/MCP)
  * memory    — course state across a multi-turn, module-by-module build     (D3 skills/memory)
  * builder   — coverage self-check drives self-correction                   (D4 quality/eval)
  * run_demo  — reproducible, observable, graceful-degradation run           (D5 production)
"""

from .provider import get_provider, CourseProvider, MockProvider, GeminiProvider
from .memory import CourseState
from .tools import CourseToolServer
from .builder import CourseArchitect, BuildReport

__all__ = [
    "get_provider", "CourseProvider", "MockProvider", "GeminiProvider",
    "CourseState", "CourseToolServer", "CourseArchitect", "BuildReport",
]


In [ ]:
%%writefile agent/provider.py
"""LLM provider: a Gemini course-designer with a deterministic Mock fallback.

Two reasoning steps, both vibe-coded as natural-language roles:
  * `plan_course`    — an instructional designer turns the outline into a module plan.
  * `generate_module`— a slide writer turns one module into content slides (multi-turn:
                       it sees the course context and the modules already written).

With a Gemini key these are real LLM calls; without one, `MockProvider` produces a
coherent, full-coverage deck deterministically, so the notebook always runs and
always emits a real `.pptx`. (Same fallback pattern as the course starter kernel.)
"""

from __future__ import annotations

import json
import os
import re
from typing import Dict, List, Optional

from deck import Outline


# --------------------------------------------------------------------------- #
# helpers
# --------------------------------------------------------------------------- #

def _short_title(text: str, max_words: int = 8) -> str:
    words = re.sub(r"\s+", " ", text.strip().rstrip(".")).split()
    t = " ".join(words[:max_words])
    return t[:1].upper() + t[1:] if t else "Overview"


def _find_api_key() -> Optional[str]:
    for var in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
        if os.environ.get(var):
            return os.environ[var]
    try:
        from kaggle_secrets import UserSecretsClient  # type: ignore
        client = UserSecretsClient()
        for var in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
            try:
                v = client.get_secret(var)
                if v:
                    return v
            except Exception:
                pass
    except Exception:
        pass
    return None


# --------------------------------------------------------------------------- #
# base
# --------------------------------------------------------------------------- #

class CourseProvider:
    name = "base"

    def generate_outline(self, topic: str) -> Outline:
        """Design a full course outline from a one-line topic."""
        raise NotImplementedError

    def plan_course(self, outline: Outline, skill: str) -> Dict:
        raise NotImplementedError

    def generate_module(self, module_plan: Dict, course_ctx: Dict,
                        skill: str, prior_titles: List[str]) -> List[Dict]:
        raise NotImplementedError


# --------------------------------------------------------------------------- #
# Mock — deterministic, full coverage, no key needed
# --------------------------------------------------------------------------- #

class MockProvider(CourseProvider):
    name = "mock"

    def generate_outline(self, topic: str) -> Outline:
        t = (topic or "").strip().rstrip(".").strip() or "a practical skill"
        title = t[:1].upper() + t[1:]
        return Outline(
            title=title,
            audience="Beginners",
            modules=[
                {"title": f"Getting started with {t}",
                 "bullets": [f"What {t} is and why it matters",
                             "The key terms and ideas to know",
                             "A simple first example"]},
                {"title": "Core techniques",
                 "bullets": ["The most common patterns step by step",
                             "A worked example to follow along",
                             "Common mistakes and how to avoid them"]},
                {"title": "Putting it into practice",
                 "bullets": ["A realistic end-to-end example",
                             "Tips to work faster and more accurately",
                             "Where to go next to keep improving"]},
            ])

    def plan_course(self, outline: Outline, skill: str) -> Dict:
        modules = []
        for m in outline.modules:
            bullets = m.get("bullets", [])
            modules.append({
                "title": m["title"],
                "objective": f"By the end, learners can apply {m['title'].lower()}.",
                "source_bullets": list(bullets),
                "slide_count": max(1, len(bullets)),
            })
        return {
            "subtitle": f"A practical guide for {outline.audience}",
            "theme": "corporate",
            "modules": modules,
        }

    def generate_module(self, module_plan: Dict, course_ctx: Dict,
                        skill: str, prior_titles: List[str]) -> List[Dict]:
        audience = course_ctx.get("audience", "your team")
        bullets = module_plan.get("source_bullets", [])
        slides: List[Dict] = []
        if not bullets:
            slides.append({
                "title": _short_title(module_plan["title"]),
                "bullets": ["What this module covers", "Why it matters", "How to apply it"],
                "notes": f"Introduce {module_plan['title']} for {audience}; set expectations.",
                "covers": [],
            })
            return slides
        for b in bullets:
            slides.append({
                "title": _short_title(b),
                "bullets": [
                    f"Key idea: {b}",
                    f"Why it matters for {audience}",
                    "A common pitfall to avoid",
                    "One step to apply it this week",
                ],
                "example": f"Walk through one real case of '{b}' step by step with the group.",
                "notes": (f"Explain '{b}' with a concrete example relevant to {audience}. "
                          f"Give one do and one don't, then check understanding."),
                "covers": [b],
            })
        return slides


# --------------------------------------------------------------------------- #
# Gemini
# --------------------------------------------------------------------------- #

_PLANNER_SYSTEM = """You are an expert instructional designer. You turn a short course \
outline into a structured module plan for a slide deck. Follow the provided skill \
rules, especially: build only from the given bullets, never invent new topics, and \
make sure every input bullet is assigned to a module.
OUTPUT STRICT JSON only:
{"subtitle": "...", "theme": "corporate|slate|fresh",
 "modules": [{"title": "...", "objective": "By the end, learners can ...",
 "source_bullets": ["...exact input bullets for this module..."], "slide_count": <int>}]}"""

_WRITER_SYSTEM = """You are an expert courseware writer. You turn ONE module into rich, \
teachable content slides. Follow the skill rules strictly: one idea per slide; 4-6 \
SUBSTANTIAL parallel bullets (each a complete, specific statement of ~8-16 words that \
carries real information — a rule, step, criterion or consequence — never vague labels); \
ONE concrete worked "example" per slide (a real mini-scenario, numbers, or do/don't a \
learner could try — one or two sentences); substantial speaker "notes" (3-5 sentences in \
the trainer's voice: what to say, a second example or analogy, a common mistake, a check \
question). Expand ONLY the given source bullets (do not invent new topics). In each \
slide's "covers", copy the exact source bullet text(s) that slide addresses, so coverage \
can be verified.
OUTPUT STRICT JSON only:
{"slides": [{"title": "...", "bullets": ["...", "..."], "example": "...", "notes": "...",
 "covers": ["...exact source bullet..."]}]}"""

_OUTLINE_SYSTEM = """You are an expert instructional designer. Given a short topic, \
design a practical course OUTLINE for it. Choose a clear, specific course title and a \
sensible target audience, then 4-5 modules that cover the topic end to end in a logical \
order. Each module has 4-5 concrete, teachable bullet points — real sub-topics a learner \
actually needs (e.g. for "basic Excel formulas": cell references, the SUM function, \
order of operations) — never vague filler like "introduction" or "why it matters".
OUTPUT STRICT JSON only:
{"title": "...", "audience": "...",
 "modules": [{"title": "...", "bullets": ["...specific point...", "..."]}]}"""

# Lighter models carry more generous free-tier quota. They are used as automatic
# fallbacks when the primary model is rate-limited, so a busy flagship downgrades
# to a slightly smaller *Gemini* model rather than to offline templated content.
_FALLBACK_MODELS = ["gemini-2.5-flash", "gemini-3.5-flash",
                    "gemini-2.5-flash-lite", "gemini-3.1-flash-lite"]


class GeminiProvider(CourseProvider):
    name = "gemini"

    def __init__(self, api_key: str, model: Optional[str] = None):
        from google import genai
        from google.genai import types
        self._genai = genai
        self._types = types
        self._client = genai.Client(api_key=api_key)
        # Default to a high free-tier-quota model (500 req/day) so casual use does
        # not hit the flagship's 20/day cap. Override with GEMINI_MODEL if desired.
        primary = model or os.environ.get("GEMINI_MODEL", "gemini-3.1-flash-lite")
        self.models = [primary] + [m for m in _FALLBACK_MODELS if m != primary]
        self._mi = 0
        self.model_name = primary
        self.degraded = 0           # steps that still fell back to offline content
        self._mock = MockProvider()

    def generate_outline(self, topic: str) -> Outline:
        prompt = (f"TOPIC: {topic}\n\nDesign the course outline as JSON. "
                  f"Make the bullets specific to this exact topic.")
        data = self._json_call(_OUTLINE_SYSTEM, prompt, max_tokens=1200)
        if not data or not data.get("modules"):
            self.degraded += 1
            return self._mock.generate_outline(topic)
        mods = []
        for m in data["modules"]:
            bl = [str(b).strip() for b in m.get("bullets", []) if str(b).strip()]
            if bl:
                mods.append({"title": str(m.get("title", "Module")).strip()[:120],
                             "bullets": bl[:6]})
        if not mods:
            self.degraded += 1
            return self._mock.generate_outline(topic)
        return Outline(title=str(data.get("title") or topic).strip()[:120],
                       audience=str(data.get("audience") or "General audience").strip()[:120],
                       modules=mods)

    def _json_call(self, system: str, prompt: str, max_tokens: int = 2048,
                   tries_per_model: int = 2) -> Optional[Dict]:
        """One JSON-returning Gemini call. Walks the model chain on rate limits, so a
        quota-capped flagship downgrades to a lighter *Gemini* model rather than to
        offline content. Returns None only if every model is unavailable (the caller
        then uses the Mock for that one step). System prompt is per-call (new SDK)."""
        import time
        config = self._types.GenerateContentConfig(
            system_instruction=system, response_mime_type="application/json",
            temperature=0.7, max_output_tokens=max_tokens)
        last = None
        mi = self._mi
        while mi < len(self.models):
            model = self.models[mi]
            delay = 2.0
            for attempt in range(tries_per_model):
                try:
                    resp = self._client.models.generate_content(
                        model=model, contents=prompt, config=config)
                    self._mi, self.model_name = mi, model     # stick with what works
                    return json.loads(resp.text)
                except Exception as exc:
                    last, msg = exc, str(exc)
                    quota = "429" in msg or "RESOURCE_EXHAUSTED" in msg
                    busy = "503" in msg or "UNAVAILABLE" in msg
                    if quota:
                        break                                 # exhausted -> switch model
                    if busy and attempt < tries_per_model - 1:
                        print(f"    [gemini] {model} busy; retry in {delay:.0f}s")
                        time.sleep(delay); delay *= 2; continue
                    if not (quota or busy):
                        print(f"    [gemini] {model}: {type(exc).__name__}; "
                              f"offline for this step")
                        return None
            nxt = mi + 1
            if nxt < len(self.models):
                print(f"    [gemini] {model} unavailable -> trying {self.models[nxt]}")
            self._mi = mi = nxt                               # advance permanently
        print(f"    [gemini] all models rate-limited "
              f"({type(last).__name__ if last else 'n/a'}); offline for this step")
        return None

    def plan_course(self, outline: Outline, skill: str) -> Dict:
        mods = "\n".join(
            f"## {m['title']}\n" + "\n".join(f"- {b}" for b in m.get("bullets", []))
            for m in outline.modules)
        prompt = (f"SKILL:\n{skill}\n\nCOURSE TITLE: {outline.title}\n"
                  f"AUDIENCE: {outline.audience}\n\nOUTLINE:\n{mods}\n\n"
                  f"Produce the module plan as JSON.")
        data = self._json_call(_PLANNER_SYSTEM, prompt)
        if not data or "modules" not in data:
            self.degraded += 1
            return self._mock.plan_course(outline, skill)
        # Ensure every module has its source bullets (fall back to the outline's).
        for i, m in enumerate(data["modules"]):
            if not m.get("source_bullets") and i < len(outline.modules):
                m["source_bullets"] = list(outline.modules[i].get("bullets", []))
        return data

    def generate_module(self, module_plan: Dict, course_ctx: Dict,
                        skill: str, prior_titles: List[str]) -> List[Dict]:
        prior = ", ".join(prior_titles) if prior_titles else "(this is the first module)"
        src = "\n".join(f"- {b}" for b in module_plan.get("source_bullets", []))
        prompt = (f"SKILL:\n{skill}\n\nCOURSE: {course_ctx.get('title')} "
                  f"(for {course_ctx.get('audience')})\n"
                  f"MODULES ALREADY WRITTEN: {prior}\n\n"
                  f"THIS MODULE: {module_plan['title']}\n"
                  f"OBJECTIVE: {module_plan.get('objective', '')}\n"
                  f"SOURCE BULLETS:\n{src}\n\nWrite this module's slides as JSON.")
        data = self._json_call(_WRITER_SYSTEM, prompt, max_tokens=4096)
        if not data or "slides" not in data or not data["slides"]:
            self.degraded += 1
            return self._mock.generate_module(module_plan, course_ctx, skill, prior_titles)
        # Coerce shape.
        out = []
        for s in data["slides"]:
            out.append({"title": str(s.get("title", "Slide"))[:120],
                        "bullets": [str(b) for b in s.get("bullets", [])][:6],
                        "example": str(s.get("example", "")),
                        "notes": str(s.get("notes", "")),
                        "covers": [str(c) for c in s.get("covers", [])]})
        return out


# --------------------------------------------------------------------------- #
# factory
# --------------------------------------------------------------------------- #

def get_provider(force_mock: bool = False) -> CourseProvider:
    if force_mock:
        print("[provider] forced Mock course designer (deterministic, no API).")
        return MockProvider()
    key = _find_api_key()
    if not key:
        print("[provider] no GOOGLE_API_KEY/GEMINI_API_KEY → Mock course designer "
              "(deterministic). Add a key to enable Gemini-written content.")
        return MockProvider()
    try:
        import google.genai  # noqa: F401
    except Exception:
        print("[provider] google-genai not installed → Mock. "
              "`pip install google-genai` to enable Gemini.")
        return MockProvider()
    prov = GeminiProvider(api_key=key)
    print(f"[provider] Gemini course designer active (model: {prov.model_name}).")
    return prov


In [ ]:
%%writefile agent/memory.py
"""Course state — the agent's working memory across a multi-turn build.

The deck is built module-by-module, and each module's generation sees the course
context and the modules already done (course Day-3 memory/state). This object holds
that growing state and can render it to a `DeckSpec` for assembly at any point. It
also persists to JSON so a build can be inspected or resumed.
"""

from __future__ import annotations

import json
import os
from typing import Dict, List, Optional

from deck import DeckSpec, Section, Slide


class CourseState:
    def __init__(self, path: Optional[str] = None):
        self.path = path
        self.title: str = "Untitled Course"
        self.subtitle: str = ""
        self.audience: str = ""
        self.theme: str = "corporate"
        self.plan: Dict = {}                       # {modules: [{title, objective, source_bullets, slide_count}]}
        self.modules: Dict[str, List[Dict]] = {}   # module_title -> [slide dicts]
        self.coverage: Dict = {}
        self.log: List[str] = []
        if path and os.path.exists(path):
            self.load()

    # -- building blocks ---------------------------------------------------- #
    def set_plan(self, plan: Dict, title: str, audience: str) -> None:
        self.plan = plan
        self.title = title
        self.audience = audience
        self.subtitle = plan.get("subtitle", "") or f"A practical guide for {audience}"
        self.theme = plan.get("theme", self.theme)
        self.save()

    def set_module(self, module_title: str, slides: List[Dict]) -> None:
        self.modules[module_title] = slides
        self.save()

    def done_module_titles(self) -> List[str]:
        return list(self.modules.keys())

    def deck_spec(self) -> DeckSpec:
        sections: List[Section] = []
        for m in self.plan.get("modules", []):
            mt = m["title"]
            slides = [Slide(title=s.get("title", ""), bullets=list(s.get("bullets", [])),
                            notes=s.get("notes", ""), example=s.get("example", ""),
                            covers=list(s.get("covers", [])))
                      for s in self.modules.get(mt, [])]
            sections.append(Section(module_title=mt, objective=m.get("objective", ""),
                                    slides=slides))
        return DeckSpec(title=self.title, subtitle=self.subtitle, audience=self.audience,
                        theme=self.theme, sections=sections)

    # -- persistence -------------------------------------------------------- #
    def save(self) -> None:
        if not self.path:
            return
        with open(self.path, "w", encoding="utf-8") as f:
            json.dump({"title": self.title, "subtitle": self.subtitle,
                       "audience": self.audience, "theme": self.theme,
                       "plan": self.plan, "modules": self.modules,
                       "coverage": self.coverage, "log": self.log}, f, indent=2)

    def load(self) -> None:
        with open(self.path, "r", encoding="utf-8") as f:
            d = json.load(f)
        self.__dict__.update({k: d.get(k, getattr(self, k)) for k in
                              ("title", "subtitle", "audience", "theme",
                               "plan", "modules", "coverage", "log")})


In [ ]:
%%writefile agent/tools.py
"""An MCP-style tool surface for the agent (course Day-2: tools & MCP).

The agent reasons (plan, write) via the LLM provider, but every *action* on the
world goes through these schema-typed tools — build the deck file, and check
coverage — with a `list_tools()` / `call_tool()` dispatcher and a logged call
history, the same shape as a real Model Context Protocol server.
"""

from __future__ import annotations

import os
from typing import Dict, List

from deck import DeckSpec, Outline, THEMES, assemble_pptx, coverage_report, get_theme


class CourseToolServer:
    def __init__(self, out_dir: str = "."):
        self.out_dir = out_dir
        self.call_log: List[Dict] = []

    def list_tools(self) -> List[Dict]:
        return [
            {"name": "assemble_deck",
             "description": "Render a DeckSpec to a .pptx file with the chosen theme. "
                            "Pure layout — adds no content of its own.",
             "inputSchema": {"type": "object",
                             "properties": {"deck": {"type": "object"},
                                            "out_path": {"type": "string"}},
                             "required": ["deck", "out_path"]}},
            {"name": "check_coverage",
             "description": "Verify every input outline bullet is represented on a slide. "
                            "Returns score, covered count, and any missing bullets.",
             "inputSchema": {"type": "object",
                             "properties": {"outline": {"type": "object"},
                                            "deck": {"type": "object"}},
                             "required": ["outline", "deck"]}},
            {"name": "list_themes",
             "description": "List available visual themes.",
             "inputSchema": {"type": "object", "properties": {}}},
        ]

    def call_tool(self, name: str, arguments: Dict) -> Dict:
        self.call_log.append({"tool": name, "arguments_keys": list(arguments.keys())})
        if name == "assemble_deck":
            return self._assemble(arguments["deck"], arguments["out_path"])
        if name == "check_coverage":
            return self._coverage(arguments["outline"], arguments["deck"])
        if name == "list_themes":
            return {"themes": list(THEMES.keys())}
        raise ValueError(f"unknown tool: {name}")

    # -- implementations ---------------------------------------------------- #
    def _assemble(self, deck_dict: Dict, out_path: str) -> Dict:
        deck = DeckSpec.from_dict(deck_dict)
        path = out_path if os.path.isabs(out_path) else os.path.join(self.out_dir, out_path)
        assemble_pptx(deck, path, get_theme(deck.theme))
        return {"path": path, "slides": deck.content_slide_count() + len(deck.sections) + 2,
                "content_slides": deck.content_slide_count(), "theme": deck.theme}

    def _coverage(self, outline_dict: Dict, deck_dict: Dict) -> Dict:
        outline = Outline(title=outline_dict.get("title", ""),
                          audience=outline_dict.get("audience", ""),
                          modules=outline_dict.get("modules", []))
        return coverage_report(outline, DeckSpec.from_dict(deck_dict))


In [ ]:
%%writefile agent/skills/instructional_design.md
# Skill: instructional design for slide courseware

Reusable expertise the agent loads before planning and writing slides (course Day-3
"skills"). Keep it short and prescriptive — it shapes every generation call.

## Scope fidelity (the golden rule)
- Build slides **only** from the bullets the user supplied. Expand and clarify them,
  but **do not invent new topics** or pad with material they didn't ask for.
- Every input bullet must end up represented on at least one slide. Record which
  input bullet(s) each slide addresses in its `covers` field.

## Structure
- One clear idea per content slide. Prefer 4–6 substantial bullets; never more than 6.
- Bullets teach, not tease: each is a complete, specific statement (roughly 8–16 words)
  that carries real information — a rule, a step, a criterion, a consequence — not a
  vague label like "overview" or "why it matters".
- Each module starts from a single learning **objective** ("By the end, learners can…").
- Bullets are parallel and action-oriented — informative phrases, not full paragraphs.
- Group closely related input bullets onto one slide; split a dense bullet into its
  own slide if it carries several distinct ideas.

## Worked example (every content slide)
- Every content slide includes an `example`: ONE concrete worked example, mini-scenario,
  or do/don't specific to that slide's idea — something a learner could actually try or
  recognise (numbers, named situations, real phrasing; not "e.g. various cases").
- Keep it to one or two sentences; it renders in a highlighted example box.

## Speaker notes
- Every content slide gets substantial speaker notes (3–5 sentences): what to say in the
  trainer's voice, a second example or analogy, a common mistake to warn about, and a
  check question or transition. Notes carry the depth; slides stay clean.

## Tone
- Match the stated audience and level. Concrete over abstract. Plain language.

## Deck shape
- Title slide → for each module: a section divider (title + objective) then its
  content slides → a closing summary slide. (The assembler adds title/section/summary
  framing; you produce the per-module content slides.)


In [ ]:
%%writefile agent/builder.py
"""The orchestrator: outline -> plan -> write modules -> assemble -> verify -> fix.

The agent loop:
  1. **Plan** (LLM): outline -> module plan (objectives, slide budget). [Day 1 vibe coding]
  2. **Write** (LLM, multi-turn): each module -> content slides, seeing the course
     context and modules already written. [Day 3 memory/state]
  3. **Assemble** (tool): render the deck to .pptx. [Day 2 tools/MCP]
  4. **Verify** (tool): coverage check — did every input bullet land on a slide? [Day 4 eval]
  5. **Self-correct**: regenerate the modules that dropped bullets, then re-verify.
The model produces all content; code only assembles and checks. [Day 5 production]
"""

from __future__ import annotations

import os
from dataclasses import dataclass, field
from typing import Dict, List, Optional

from deck import Outline, parse_outline

from .memory import CourseState
from .provider import CourseProvider, get_provider
from .tools import CourseToolServer

_SKILL_PATH = os.path.join(os.path.dirname(__file__), "skills", "instructional_design.md")


def _load_skill() -> str:
    try:
        with open(_SKILL_PATH, "r", encoding="utf-8") as f:
            return f.read()
    except OSError:
        return "(skill file unavailable)"


def _outline_dict(o: Outline) -> Dict:
    return {"title": o.title, "audience": o.audience, "modules": o.modules}


@dataclass
class BuildReport:
    title: str
    provider: str
    theme: str
    modules: int
    content_slides: int
    total_slides: int
    coverage_score: float
    missing: List[str]
    out_path: str
    tool_calls: int
    fix_rounds: int
    log: List[str] = field(default_factory=list)

    def headline(self) -> str:
        cov = f"{self.coverage_score:.0%} coverage"
        gap = "all bullets covered" if not self.missing else f"{len(self.missing)} bullet(s) still missing"
        return (f"{self.provider} built '{self.title}': {self.total_slides} slides "
                f"({self.content_slides} content) across {self.modules} modules, "
                f"{cov} ({gap}). Saved to {self.out_path}.")


class CourseArchitect:
    def __init__(self, provider: Optional[CourseProvider] = None, theme: Optional[str] = None,
                 state_path: Optional[str] = None, out_dir: str = ".", verbose: bool = True):
        self.provider = provider or get_provider()
        self.theme_override = theme
        self.tools = CourseToolServer(out_dir=out_dir)
        self.state = CourseState(state_path)
        self.skill = _load_skill()
        self.verbose = verbose
        self.log: List[str] = []

    def _log(self, msg: str) -> None:
        self.log.append(msg)
        if self.verbose:
            print(msg)

    def build(self, outline_text: str, out_name: str = "course.pptx",
              max_fix_rounds: int = 1) -> BuildReport:
        # If the input gives no "## module" headings, treat it as a plain-English
        # topic and let the agent DESIGN the outline before writing the content.
        has_modules = any(l.strip().startswith("## ") for l in outline_text.splitlines())
        if has_modules:
            outline = parse_outline(outline_text)
        else:
            topic = self._extract_topic(outline_text)
            self._log(f"[outline] no sections supplied — {self.provider.name} is "
                      f"designing the outline for: {topic!r}")
            outline = self.provider.generate_outline(topic)
        self._log(f"[plan] '{outline.title}' for {outline.audience} — "
                  f"{len(outline.modules)} modules, {len(outline.all_bullets())} bullets")

        # 1. Plan
        plan = self.provider.plan_course(outline, self.skill)
        self.state.set_plan(plan, outline.title, outline.audience)
        if self.theme_override:
            self.state.theme = self.theme_override
        self._log(f"[plan] {self.provider.name} -> {len(plan.get('modules', []))} modules, "
                  f"theme '{self.state.theme}'")

        # 2. Write each module (multi-turn: pass titles already written)
        ctx = {"title": self.state.title, "audience": self.state.audience,
               "subtitle": self.state.subtitle}
        for m in plan.get("modules", []):
            slides = self.provider.generate_module(m, ctx, self.skill,
                                                   self.state.done_module_titles())
            self.state.set_module(m["title"], slides)
            self._log(f"[write] {m['title']} -> {len(slides)} slides")

        # 3 + 4. Assemble + verify coverage
        coverage = self._verify(outline)
        self._log(f"[eval] coverage {coverage['score']:.0%} "
                  f"({coverage['covered']}/{coverage['total_bullets']} bullets)")

        # 5. Self-correct dropped bullets
        fixes = 0
        while coverage["missing"] and fixes < max_fix_rounds:
            fixes += 1
            self._fix_missing(coverage["missing"], ctx)
            coverage = self._verify(outline)
            self._log(f"[fix {fixes}] re-coverage {coverage['score']:.0%} "
                      f"({len(coverage['missing'])} missing)")

        # Final assemble via the tool
        deck = self.state.deck_spec()
        res = self.tools.call_tool("assemble_deck",
                                   {"deck": deck.to_dict(), "out_path": out_name})
        self.state.coverage = coverage
        self.state.save()

        report = BuildReport(
            title=self.state.title, provider=self.provider.name, theme=self.state.theme,
            modules=len(deck.sections), content_slides=res["content_slides"],
            total_slides=res["slides"], coverage_score=coverage["score"],
            missing=coverage["missing"], out_path=res["path"],
            tool_calls=len(self.tools.call_log), fix_rounds=fixes, log=list(self.log))
        self._log("\n" + report.headline())
        return report

    # -- internals ---------------------------------------------------------- #
    @staticmethod
    def _extract_topic(text: str) -> str:
        """Pull a plain-English topic out of free text (ignoring template hints,
        markdown markers and an 'Audience:' line)."""
        parts: List[str] = []
        for raw in text.splitlines():
            s = raw.strip()
            if not s:
                continue
            if s.startswith("(") and s.endswith(")"):   # template hint line
                continue
            if s.lower().startswith("audience:"):
                continue
            s = s.lstrip("#-*• ").strip()                # drop md markers
            if s:
                parts.append(s)
        return " ".join(parts).strip() or "a useful introductory course"

    def _verify(self, outline: Outline) -> Dict:
        deck = self.state.deck_spec()
        return self.tools.call_tool("check_coverage",
                                    {"outline": _outline_dict(outline), "deck": deck.to_dict()})

    def _fix_missing(self, missing: List[str], ctx: Dict) -> None:
        """Regenerate, per owning module, slides for the bullets that got dropped, and
        append them — using the provider (never code) to produce the content."""
        missing_set = set(missing)
        for m in self.state.plan.get("modules", []):
            owned = [b for b in m.get("source_bullets", []) if b in missing_set]
            if not owned:
                continue
            synthetic = {"title": m["title"], "objective": m.get("objective", ""),
                         "source_bullets": owned}
            extra = self.provider.generate_module(synthetic, ctx, self.skill,
                                                  self.state.done_module_titles())
            self.state.modules[m["title"]] = self.state.modules.get(m["title"], []) + extra
        self.state.save()


## 2. (Optional) Enable Gemini-written content
Skip to run the deterministic offline writer. To use Gemini:
1. Add your AI Studio key as a Kaggle **Secret** named `GOOGLE_API_KEY` (Add-ons → Secrets), or
2. set it in-notebook: `import os; os.environ["GOOGLE_API_KEY"] = "..."`

Model defaults to `gemini-3.1-flash-lite` (generous free-tier quota); override with
`GEMINI_MODEL`. If a model is rate-limited, the agent automatically falls back across a
chain of Gemini models — never to offline text — before degrading. A small production touch.

## 3. Your course outline
The agent's input: a title, an audience, and a few modules with bullet points. Edit freely.

In [ ]:
OUTLINE = """# Giving Effective Feedback
Audience: Newly promoted team leaders (beginner)

## Why feedback matters
- The hidden cost of staying silent about problems
- Feedback as a tool for building trust, not just correcting
- The difference between feedback and criticism

## A simple model for feedback
- The Situation-Behaviour-Impact (SBI) model
- Keeping feedback specific and observable
- Focusing on behaviour, not personality

## Handling difficult reactions
- Staying calm when feedback is met with defensiveness
- Listening before responding
- Agreeing on a concrete next step together
"""
print(OUTLINE)

## 4. Build the course
Plan → write each module (multi-turn) → assemble the `.pptx` → check coverage → self-correct
any dropped points. Watch the log.

In [ ]:
from agent import CourseArchitect, get_provider

provider = get_provider()          # Gemini if a key is present, else offline writer
architect = CourseArchitect(provider=provider, theme="corporate",
                            state_path="course_state.json", out_dir=".")
report = architect.build(OUTLINE, out_name="course.pptx")
print()
print(report.headline())

## 5. Preview the slides
Kaggle can't render `.pptx` inline, so we draw a few slides from the deck spec with the
theme's colours. The real, editable file is `course.pptx` in the notebook output.

In [ ]:
import matplotlib.pyplot as plt
from deck import get_theme

def _hex(h): return "#" + h
def render_slides(deck, n=3):
    th = get_theme(deck.theme)
    panels = [("title", deck.title, [deck.subtitle or ("For: " + deck.audience)], th.band, "FFFFFF")]
    for sec in deck.sections:
        for sl in sec.slides:
            panels.append(("content", sl.title, sl.bullets, th.bg, th.title))
    panels = panels[:n+1]
    fig, axes = plt.subplots(1, len(panels), figsize=(5.2*len(panels), 3.4))
    if len(panels) == 1: axes = [axes]
    for ax, (kind, title, bullets, bg, tcol) in zip(axes, panels):
        ax.set_xlim(0,10); ax.set_ylim(0,6); ax.axis("off")
        ax.add_patch(plt.Rectangle((0,0),10,6, color=_hex(bg)))
        ax.add_patch(plt.Rectangle((0,5.7),10,0.3, color=_hex(th.accent)))
        ax.text(0.4, 5.0, title, fontsize=13, fontweight="bold", color=_hex(tcol), wrap=True)
        for i,b in enumerate(bullets[:5]):
            ax.text(0.6, 4.2-0.6*i, "• "+b, fontsize=9, color=_hex(th.body if kind=="content" else "FFFFFF"), wrap=True)
    plt.tight_layout(); plt.show()

render_slides(architect.state.deck_spec(), n=3)

## 6. The agent can design the outline itself
No outline? Give it a one-line **topic** and the agent designs the whole curriculum — title,
audience, modules and teachable bullet points — which flows straight into the same build
pipeline. (This is exactly what the one-click 'Describe a Course' launcher uses.)

In [ ]:
# One plain-English topic in; a full structured outline out (a single Gemini call).
topic = "How to write basic Excel formulas for office staff"
designed = provider.generate_outline(topic)

print(f"TOPIC: {topic}\n")
print(f"-> '{designed.title}'  (audience: {designed.audience})")
for i, m in enumerate(designed.modules, 1):
    print(f"\n  Module {i}: {m['title']}")
    for b in m["bullets"]:
        print(f"    - {b}")
print("\nGive that topic to architect.build(topic) and it designs the outline, then writes the deck.")

## 7. The MCP tool surface & the coverage check
The agent reasons with the LLM but only *acts* through these schema-typed tools — the same
shape as a Model Context Protocol server.

In [ ]:
import json
from agent import CourseToolServer
ts = CourseToolServer(out_dir=".")
print(json.dumps(ts.list_tools(), indent=2)[:1100], "...")
print("\nFinal coverage scorecard:")
print(json.dumps(architect.state.coverage, indent=2)[:900])

## 8. How it works — architecture & design

```
   (TOPIC ──▶ [ Outline Designer (Gemini) ] ──▶ OUTLINE)   ← optional: if you give only a topic
   OUTLINE ──▶ [ Planner (Gemini) ] ──▶ module plan ──▶ [ Writer (Gemini), per module ]
                                                                  │ multi-turn: sees course
                                                                  │ context + modules done
                                                                  ▼
   .pptx ◀── [ assemble_deck TOOL ] ◀── DeckSpec ◀── content slides (+ speaker notes)
                                            │
                                            ▼
                                  [ check_coverage TOOL ] ──▶ missing bullets?
                                            │ yes → regenerate those modules (self-correct)
                                            ▼ no
                                        DONE ✓
```

**Designs its own curriculum.** Given only a topic, the agent first designs a full outline
(modules + teachable points) before building — so a non-expert gets a structured course from
a single sentence, while an expert can still hand it a precise outline to follow exactly.

**Model writes, code assembles.** The LLM produces every title, bullet and speaker note;
`python-pptx` only lays them out and applies the theme. Code never invents content — a
production discipline that keeps output faithful and themes swappable (brand applied last).

**Scope fidelity, enforced by evaluation.** The agent must cover *only* the bullets you gave —
no padding, nothing dropped. The `check_coverage` tool verifies every input bullet landed on
a slide and feeds any gaps back into a **self-correction** round. That objective self-check is
the project's backbone, and the habit that matters most in real agent systems: *don't just
generate — verify, then fix.*

**Memory across a multi-turn build.** Slides are written module-by-module; each call sees the
course context and the modules already written, so the deck stays coherent rather than a bag
of disconnected slides.

**Graceful degradation (production thinking).** No key → deterministic offline writer.
Unparseable model output → coerced/fallback content. The build never stalls; the notebook
always emits a real `.pptx`.

### Why this matters (Agents for Business)
Turning rough outlines into structured, on-brand courseware is real, repetitive knowledge
work. The same pattern — *plan → generate section-by-section → assemble a real document →
verify coverage → self-correct* — generalises to any structured-document generation: reports,
proposals, onboarding packs, lesson plans.

### Limitations & next steps
- Layout is template-based (title / section / content / summary); richer layouts (two-column,
  image slides, diagrams) and an image-sourcing tool are natural extensions.
- Coverage is lexical (token overlap); an LLM-as-judge rubric (clarity, accuracy, tone) would
  deepen the Day-4 evaluation.
- A human-in-the-loop review/approve step per module would suit enterprise use.

*Built for the Google × Kaggle 5-Day AI Agents Intensive (Vibe Coding) capstone. Themes and
prompts here are generic; no proprietary brand assets are included.*